In [0]:
from pyspark.sql import functions as F

In [0]:
ranked_preds = spark.table("renewal_model_ranked_predictions")

renewal_dashboard_base = (
    ranked_preds
    .withColumn(
        "revenue_at_risk",
        F.col("annual_contract_value") * F.col("non_renewal_probability")
        # Calculation of Expected Loss, to have an idea of which accounts to prioritize between large accounts with moderate risk of churning vs smaller accounts with extremely high risk of churning
    )
)


In [0]:
renewal_dashboard_base.columns

In [0]:
renewal_dashboard_base.agg(F.min(F.col("active_members_change_3m_vs_prior_3m"))).show()

In [0]:
# Logic created based on business rules for explainability, not causal truth
renewal_dashboard_base = (
    renewal_dashboard_base
    .withColumn(
        "driver_low_utilization",
        F.when(F.col("utilization_avg_last_3m") < 0.08, 1).otherwise(0)
    )
    .withColumn(
        "driver_utilization_drop",
        F.when(F.col("utilization_change_3m_vs_prior_3m") < -0.01, 1).otherwise(0)
    )
    .withColumn(
        "driver_active_members_drop",
        F.when(F.col("active_members_change_3m_vs_prior_3m") < -100, 1).otherwise(0)
    )
    .withColumn(
        "driver_sessions_drop",
        F.when(F.col("sessions_change_3m_vs_prior_3m") < -100, 1).otherwise(0)
    )
    .withColumn(
        "driver_nps_drop",
        F.when(F.col("nps_change_3m_vs_prior_3m") < -5, 1).otherwise(0)
    )
    .withColumn(
        "driver_high_ticket_burden",
        F.when(F.col("ticket_count_avg_last_3m") > 5, 1).otherwise(0)
    )
    .withColumn(
        "driver_high_severity_ticket_rise",
        F.when(F.col("high_severity_tickets_change_3m_vs_prior_3m") > 1, 1).otherwise(0)
    )
    .withColumn(
        "driver_low_webinar_engagement",
        F.when(F.col("webinar_attendance_avg_last_3m") < 1, 1).otherwise(0)
    )
    .withColumn(
        "driver_low_employer_comms",
        F.when(F.col("employer_comms_avg_last_3m") < 1, 1).otherwise(0)
    )
)

In [0]:
renewal_dashboard_base = (
    renewal_dashboard_base
    .withColumn(
        "top_driver_summary",
        F.concat_ws(
            ", ",
            F.when(F.col("driver_low_utilization") == 1, F.lit("Low utilization")),
            F.when(F.col("driver_utilization_drop") == 1, F.lit("Usage declining")),
            F.when(F.col("driver_active_members_drop") == 1, F.lit("Active members dropping")),
            F.when(F.col("driver_sessions_drop") == 1, F.lit("Sessions dropping")),
            F.when(F.col("driver_nps_drop") == 1, F.lit("NPS declining")),
            F.when(F.col("driver_high_ticket_burden") == 1, F.lit("High ticket burden")),
            F.when(F.col("driver_high_severity_ticket_rise") == 1, F.lit("Severe tickets rising")),
            F.when(F.col("driver_low_webinar_engagement") == 1, F.lit("Low webinar engagement")),
            F.when(F.col("driver_low_employer_comms") == 1, F.lit("Low employer communications"))
        )
    )
)

display(
    renewal_dashboard_base.select(
        "client_id",
        "non_renewal_probability",
        "risk_rank",
        "top_driver_summary"
    ).limit(20)
)

In [0]:
renewal_dashboard_base = (
    renewal_dashboard_base
    .withColumn(
        "recommended_action",
        F.when(
            (F.col("driver_nps_drop") == 1) & (F.col("driver_high_ticket_burden") == 1),
            F.lit("Escalate to CS + Support leadership")
        ).when(
            (F.col("driver_low_utilization") == 1) | (F.col("driver_sessions_drop") == 1),
            F.lit("Launch adoption recovery plan")
        ).when(
            F.col("driver_low_employer_comms") == 1,
            F.lit("Increase employer outreach and enablement")
        ).when(
            F.col("driver_low_webinar_engagement") == 1,
            F.lit("Re-engage through training/webinars")
        ).otherwise(
            F.lit("Review account health manually")
        )
    )
)

display(
    renewal_dashboard_base.select(
        "client_id",
        "non_renewal_probability",
        "risk_rank",
        "recommended_action"
    ).limit(20)
)

In [0]:
spark.sql("DROP TABLE IF EXISTS renewal_dashboard_base")

renewal_dashboard_base.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("renewal_dashboard_base")

display(renewal_dashboard_base)

In [0]:
# Build Driver Detail table for drill-down
driver_detail = (
    renewal_dashboard_base
    .select(
        "client_id",
        "renewal_cycle_id",
        "prediction_month",
        "contract_end_date",
        "non_renewal_probability",
        "annual_contract_value",
        "revenue_at_risk",
        F.expr("""
            stack(
                9,
                'Low utilization', driver_low_utilization,
                'Usage declining', driver_utilization_drop,
                'Active members dropping', driver_active_members_drop,
                'Sessions dropping', driver_sessions_drop,
                'NPS declining', driver_nps_drop,
                'High ticket burden', driver_high_ticket_burden,
                'Severe tickets rising', driver_high_severity_ticket_rise,
                'Low webinar engagement', driver_low_webinar_engagement,
                'Low employer communications', driver_low_employer_comms
            ) as (driver_name, driver_flag)
        """)
    )
    .filter(F.col("driver_flag") == 1)
)

In [0]:
display(driver_detail)

In [0]:
driver_detail = (
    driver_detail
    .withColumn(
        "driver_explanation",
        F.when(F.col("driver_name") == "Low utilization", F.lit("Recent utilization is below expected healthy level"))
         .when(F.col("driver_name") == "Usage declining", F.lit("Utilization has worsened versus the prior quarter"))
         .when(F.col("driver_name") == "Active members dropping", F.lit("Adoption is falling versus the prior quarter"))
         .when(F.col("driver_name") == "Sessions dropping", F.lit("Platform engagement is declining"))
         .when(F.col("driver_name") == "NPS declining", F.lit("Customer satisfaction has deteriorated"))
         .when(F.col("driver_name") == "High ticket burden", F.lit("Support volume is elevated"))
         .when(F.col("driver_name") == "Severe tickets rising", F.lit("High-severity support issues are increasing"))
         .when(F.col("driver_name") == "Low webinar engagement", F.lit("Training/enablement participation is weak"))
         .when(F.col("driver_name") == "Low employer communications", F.lit("Account outreach activity appears low"))
         .otherwise(F.lit("Health signal requires review"))
    )
)

In [0]:
driver_detail = (
    driver_detail
    .withColumn(
        "driver_priority",
        F.when(F.col("driver_name") == "NPS declining", 1)
         .when(F.col("driver_name") == "Severe tickets rising", 2)
         .when(F.col("driver_name") == "High ticket burden", 3)
         .when(F.col("driver_name") == "Usage declining", 4)
         .when(F.col("driver_name") == "Low utilization", 5)
         .when(F.col("driver_name") == "Active members dropping", 6)
         .when(F.col("driver_name") == "Sessions dropping", 7)
         .when(F.col("driver_name") == "Low webinar engagement", 8)
         .when(F.col("driver_name") == "Low employer communications", 9)
         .otherwise(99)
    )
)

In [0]:
spark.sql("DROP TABLE IF EXISTS renewal_account_driver_detail")

(
    driver_detail
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("renewal_account_driver_detail")
)

In [0]:
display(
    spark.table("renewal_account_driver_detail")
         .orderBy("client_id", "driver_priority")
         .limit(50)
)

In [0]:
top_5_at_risk = (
    spark.table("renewal_dashboard_base")
         .orderBy(F.asc("risk_rank"))
         .limit(5)
)

spark.sql("DROP TABLE IF EXISTS renewal_top_5_at_risk")

(
    top_5_at_risk
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("renewal_top_5_at_risk")
)

display(spark.table("renewal_top_5_at_risk"))

In [0]:
# Option B Databricks SQL / Lakeview Dashboard 